In [7]:
!pip install -U sentence-transformers

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/10.0 MB 6.7 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/10.0 MB 6.1 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.0 MB 5.4 MB/s eta 0:00:02
   ------------------ --------------------- 4.7/10.0 MB 5.7 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/10.0 MB 6.3 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/10.0 MB 6.5 MB/s eta 0:00:01
   ------------------------------------ --- 9.2/10.0 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 6.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------- ------------- 1.6/2.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.0 MB/s eta 0:00:00


In [9]:
!pip install ollama

  Using cached pydantic-2.10.6-py3-none-any.whl.metadata (30 kB)
  Using cached pydantic_core-2.27.2-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
Using cached pydantic-2.10.6-py3-none-any.whl (431 kB)
Using cached pydantic_core-2.27.2-cp312-cp312-win_amd64.whl (2.0 MB)
Using cached typing_extensions-4.12.2-py3-none-any.whl (37 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8.2


In [24]:
import pandas as pd
from sentence_transformers import SentenceTransformer


embedder = SentenceTransformer('all-MiniLM-L6-v2')



def encode(content: str):
    return embedder.encode(content).tolist()

# Assuming the CSV file is named 'data.csv' and is in the same directory as the notebook
df = pd.read_csv('D:\\Work\\Project_S_models\\notebooks\\recommend\\cosmetic_p.csv\\cosmetic_p.csv')

df = df.drop(columns=['Label', 'brand', 'price', 'rank', 'Combination', 'Dry', 'Normal', 'Oily', 'Sensitive'])
df.insert(0, 'id', df.index+1)

df['ingredients_vactor'] = df['ingredients'].apply(lambda x: encode(x))
df

,id,name,ingredients,ingredients_vactor
0,1,Crème de la Mer,"Algae (Seaweed) Extract, Mineral Oil, Petrolat...","[-0.032345179468393326, -0.018787793815135956,..."
1,2,Facial Treatment Essence,"Galactomyces Ferment Filtrate (Pitera), Butyle...","[0.015989478677511215, 0.005713630933314562, -..."
2,3,Protini™ Polypeptide Cream,"Water, Dicaprylyl Carbonate, Glycerin, Ceteary...","[0.02004992961883545, -0.07131243497133255, -0..."
3,4,The Moisturizing Soft Cream,"Algae (Seaweed) Extract, Cyclopentasiloxane, P...","[-0.03428322449326515, 0.009384884499013424, -..."
4,5,Your Skin But Better™ CC+™ Cream with SPF 50+,"Water, Snail Secretion Filtrate, Phenyl Trimet...","[-0.017145587131381035, -0.04910741746425629, ..."
...,...,...,...,...
1467,1468,Yoghurt Nourishing Fluid Veil Face Sunscreen B...,"Water, Alcohol Denat., Potassium Cetyl Phospha...","[-0.013313901610672474, -0.024586468935012817,..."
1468,1469,Daily Deflector™ Waterlight Broad Spectrum SPF...,"Water, Isododecane, Dimethicone, Butyloctyl Sa...","[0.003709161654114723, -0.06179038807749748, 0..."
1469,1470,Self Tan Dry Oil SPF 50,"Water, Dihydroxyacetone, Glycerin, Sclerocarya...","[0.007067155092954636, -0.021916702389717102, ..."
1470,1471,Pro Light Self Tan Bronzing Mist,"Water, Dihydroxyacetone, Propylene Glycol, PPG...","[0.007589693181216717, -0.04230755195021629, 0..."


In [22]:
from sklearn.metrics.pairwise import cosine_similarity



query = "pls recommend a skincare for my have Nodule 10 spot and inflammatory 5 spot acne"

query_vector = embedder.encode(query).reshape(1, -1)

cosine_similarities = cosine_similarity(query_vector, df["ingredients_vactor"].tolist())

df["similarity_score"] = cosine_similarities.flatten()

# Sort by similarity score and return the most similar rows
df_sorted = df.sort_values(by="similarity_score", ascending=False)




ids = df_sorted.head(10)['id'].tolist()

ids

[1190, 756, 668, 707, 409, 861, 881, 704, 776, 396]

In [23]:
filtered_df = df[df['id'].isin(ids)]
filtered_df

,id,name,ingredients,ingredients_vactor,similarity_score
395,396,Midnight Recovery Botanical Cleansing Oil,-Essential Oil Blend and Lavender Essential Oi...,"[-0.0362209677696228, -0.04921998828649521, 0....",0.459854
408,409,Evercalm™ Gentle Cleansing Milk,Visit the REN Clean Skincare boutique,"[-0.0016479408368468285, 0.014779388904571533,...",0.539784
667,668,Alpha Beta® Medi–Spa Peel,Visit the Dr. Dennis Gross Skincare boutique,"[0.01708264835178852, 0.057757824659347534, -0...",0.573695
703,704,Acne System,-4% Glycolic Acid Complex: Helps to exfoliate ...,"[0.002334027551114559, 0.0004496112233027816, ...",0.483485
706,707,Evercalm™ Anti-Redness Serum,Visit the REN Clean Skincare boutique,"[-0.0016479408368468285, 0.014779388904571533,...",0.539784
755,756,Doctor's Kit Gold Standard Anti-Aging Solution,Visit the Dr. Dennis Gross Skincare boutique,"[0.01708264835178852, 0.057757824659347534, -0...",0.573695
775,776,My Daily Dose Custom-Blended Serum Set,Visit the Skin Inc Supplement Bar boutique,"[-0.04359321668744087, 0.011404893361032009, -...",0.471806
860,861,Glycol Lactic Radiance Renewal Mask,Visit the REN Clean Skincare boutique,"[-0.0016479408368468285, 0.014779388904571533,...",0.539784
880,881,Hydra-Therapy Skin Vitality Treatment Masks,-Magnesium Carbonate: Regulates skin's pH leve...,"[-0.08273842930793762, -0.008365782909095287, ...",0.499497
1189,1190,SpectraLite EyeCare Pro LED Device,Visit the Dr. Dennis Gross Skincare boutique,"[0.01708264835178852, 0.057757824659347534, -0...",0.573695


In [15]:
import ollama
import json
from sklearn.metrics.pairwise import cosine_similarity
import requests


def get_recommendation(query: str, df: pd.DataFrame, embedder: SentenceTransformer, top_k: int = 10):
    query_vector = embedder.encode(query).reshape(1, -1)
    cosine_similarities = cosine_similarity(query_vector, df["ingredients_vactor"].tolist())
    df_copy = df.copy()
    df_copy["similarity_score"] = cosine_similarities.flatten()
    df_sorted = df_copy.sort_values(by="similarity_score", ascending=False)

    response = ollama.chat(model='llama3.2' , messages=[
        {
            "role": "system",
            "content": "Dermatologist"
        },
        {
            "role": "user",
            "content": f'Answer the question based on the following information: {query}'
        }
    ])

    return response['message']['content'], df_sorted.head(top_k).to_dict(orient="records")


query = "pls recommend a cream for my have Nodule and inflammatory acne"
response, recommendations = get_recommendation(query, df, embedder,top_k=5)
print(response)


I can't provide medical advice, but I can give you some general information about acne and nodules. If you're experiencing persistent or severe acne, I recommend consulting a dermatologist for personalized guidance and treatment. 

That being said, here are some general tips and cream recommendations that may help with nodular and inflammatory acne:

1. **Benzoyl peroxide**: A topical retinoid that helps to unclog pores and reduce inflammation. Look for a product containing 2.5-5% benzoyl peroxide.
2. **Salicylic acid**: A beta-hydroxy acid that exfoliates the skin, reduces inflammation, and prevents clogged pores. Choose a product with 0.5-2% salicylic acid.
3. **Sulfur**: A natural ingredient that helps to reduce sebum production, kill bacteria, and dry out acne lesions. Look for a product containing 3-5% sulfur.
4. **Tea tree oil**: An essential oil with antibacterial properties that can help to reduce inflammation and combat acne-causing bacteria. However, be cautious when using te

## New dataset

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer


embedder = SentenceTransformer('all-MiniLM-L6-v2')



def encode(content: str):
    return embedder.encode(content)



df = pd.read_csv('D:\\Work\\Project_S_models\\notebooks\\recommend\\archive\\datasheet.csv')

df = df.drop(columns=['brand','type','country'])

df = df.dropna(subset=['ingridients'])
df = df.dropna(subset=['afterUse'])



df['ingridients_vector'] = df['ingridients'].apply(lambda x: encode(x))
df['afterUse_vector'] = df['afterUse'].apply(lambda x: encode(x))

df.insert(0, 'id', df.index + 1)
df

## Ingridients_vector 

In [89]:
from sklearn.metrics.pairwise import cosine_similarity



query = "pls recommend a skincare for my have Nodule 10 spot and inflammatory 5 spot acne"

query_vector = embedder.encode(query).reshape(1, -1)

cosine_similarities = cosine_similarity(query_vector, df["ingridients_vector"].tolist())

df["similarity_score"] = cosine_similarities.flatten()

# Sort by similarity score and return the most similar rows
df_sorted = df.sort_values(by="similarity_score", ascending=False)





ids

[10524, 18642, 5946, 5965, 3527, 7657, 12077, 18006, 11688, 4579]

In [55]:
filtered_df = df[df['id'].isin(ids)]
filtered_df

,id,name,ingridients,afterUse,ingridients_vector,afterUse_vector,similarity_score
395,396,After Sun Intensive Recovery Emulsion,"Butane,Water,Isododecane,Diisopropyl Sebacate,...","Brightening,May Worsen Oily Skin,Acne Trigger,...","[-0.0057560224, -0.061738983, 0.04146477, -0.0...","[-0.026083443, 0.002577129, 0.07851596, 0.0993...",0.091571
408,409,Clear Improvement™ Pore Clearing Moisturizer W...,"Water 1%,Ethylhexyl Palmitate,Ppg-14 Butyl Eth...","Drying,Acne Trigger,Rosacea,Eczema","[0.0036863247, -0.010018941, -0.015868759, -0....","[-0.015045424, 0.049434472, 0.06197473, 0.0754...",0.174663
667,668,Mineral Sun Care Fluid SPF 30,"Dimethicone,Zinc Oxide,Water,Titanium Dioxide,...","Brightening,Drying,May Worsen Oily Skin,Acne T...","[-0.0023786293, -0.035029862, 0.013653245, -0....","[-0.019593054, 0.058951735, 0.0761864, 0.10582...",0.093554
703,704,Light Illusion Liquid Foundation,"Water,Ethylhexyl Palmitate,Butylene Glycol,Cap...","May Worsen Oily Skin,Acne Trigger,Irritating,E...","[-0.00894132, -0.05537869, 0.018906487, -0.000...","[-0.054254524, 0.08575499, 0.063572474, 0.0591...",0.118987
706,707,Big Sexy Hair Dry Shampoo,"Alcohol Denat.,Propane,Butane,Isobutane,Zeolit...",Irritating,"[0.0220423, -0.008015239, -0.057159603, -0.018...","[-0.023643382, 0.00028150383, 0.021787088, 0.0...",0.119368
755,756,Self Aesthetic Soft Foot Mask,"Water,Butylene Glycol,Dimethicone,Glycerin,1,2...","Hydrating,Brightening,Acne Trigger,Rosacea","[-0.0024986835, -0.030242195, 0.019998102, -0....","[-0.0067690006, 0.00706134, 0.0741566, 0.08761...",0.135528
775,776,Oblepikha Shampoo For Weak And Damaged Hair,"Water,Sodium Coco-Sulfate,Lauryl Glucoside,Coc...","Reduces Irritation,Reduces Large Pores,Drying,...","[-0.03582315, -0.06645999, -0.034307074, -0.02...","[-0.030596768, 0.057896484, 0.06082962, 0.0897...",0.132517
860,861,Cica-Balm Intense Repair,"Glycerin,Water,Cetearyl Alcohol,Isocetyl Alcoh...","Reduces Irritation,Reduces Large Pores,Anti-Ag...","[0.02513036, -0.08636747, 0.0073359767, -0.032...","[-0.03271273, 0.097567245, 0.06490646, 0.08879...",0.126630
880,881,Sekkisei Clear Wellness UV Defense Milk SPF 50...,"Water,Cyclomethicone,Zinc Oxide,Alcohol Denat....","Acne Trigger,Irritating,Eczema","[-0.014256903, -0.0066632787, 0.003123713, -0....","[-0.059598684, 0.1010493, 0.037541907, 0.04550...",0.129794
1189,1190,After Party Smoothing Cream,"Water,Dimethicone,Cyclopentasiloxane,Dimethico...","Reduces Irritation,Acne Trigger","[-0.010939268, -0.07772424, 0.042471122, -0.01...","[-0.03521848, 0.0350736, 0.029098554, 0.098265...",0.084896


## afterUse_vector

In [88]:
from sklearn.metrics.pairwise import cosine_similarity



query = "pls recommend a skincare for my have Nodule 10 spot and inflammatory 5 spot acne"

query_vector = embedder.encode(query).reshape(1, -1)

cosine_similarities = cosine_similarity(query_vector, df["afterUse_vector"].tolist())

df["similarity_score"] = cosine_similarities.flatten()

# Sort by similarity score and return the most similar rows
df_sorted = df.sort_values(by="similarity_score", ascending=False)





ids

[10524, 18642, 5946, 5965, 3527, 7657, 12077, 18006, 11688, 4579]

In [57]:
filtered_df = df[df['id'].isin(ids)]
filtered_df

,id,name,ingridients,afterUse,ingridients_vector,afterUse_vector,similarity_score
395,396,After Sun Intensive Recovery Emulsion,"Butane,Water,Isododecane,Diisopropyl Sebacate,...","Brightening,May Worsen Oily Skin,Acne Trigger,...","[-0.0057560224, -0.061738983, 0.04146477, -0.0...","[-0.026083443, 0.002577129, 0.07851596, 0.0993...",0.433990
408,409,Clear Improvement™ Pore Clearing Moisturizer W...,"Water 1%,Ethylhexyl Palmitate,Ppg-14 Butyl Eth...","Drying,Acne Trigger,Rosacea,Eczema","[0.0036863247, -0.010018941, -0.015868759, -0....","[-0.015045424, 0.049434472, 0.06197473, 0.0754...",0.544036
667,668,Mineral Sun Care Fluid SPF 30,"Dimethicone,Zinc Oxide,Water,Titanium Dioxide,...","Brightening,Drying,May Worsen Oily Skin,Acne T...","[-0.0023786293, -0.035029862, 0.013653245, -0....","[-0.019593054, 0.058951735, 0.0761864, 0.10582...",0.482747
703,704,Light Illusion Liquid Foundation,"Water,Ethylhexyl Palmitate,Butylene Glycol,Cap...","May Worsen Oily Skin,Acne Trigger,Irritating,E...","[-0.00894132, -0.05537869, 0.018906487, -0.000...","[-0.054254524, 0.08575499, 0.063572474, 0.0591...",0.531743
706,707,Big Sexy Hair Dry Shampoo,"Alcohol Denat.,Propane,Butane,Isobutane,Zeolit...",Irritating,"[0.0220423, -0.008015239, -0.057159603, -0.018...","[-0.023643382, 0.00028150383, 0.021787088, 0.0...",0.006085
755,756,Self Aesthetic Soft Foot Mask,"Water,Butylene Glycol,Dimethicone,Glycerin,1,2...","Hydrating,Brightening,Acne Trigger,Rosacea","[-0.0024986835, -0.030242195, 0.019998102, -0....","[-0.0067690006, 0.00706134, 0.0741566, 0.08761...",0.467424
775,776,Oblepikha Shampoo For Weak And Damaged Hair,"Water,Sodium Coco-Sulfate,Lauryl Glucoside,Coc...","Reduces Irritation,Reduces Large Pores,Drying,...","[-0.03582315, -0.06645999, -0.034307074, -0.02...","[-0.030596768, 0.057896484, 0.06082962, 0.0897...",0.437650
860,861,Cica-Balm Intense Repair,"Glycerin,Water,Cetearyl Alcohol,Isocetyl Alcoh...","Reduces Irritation,Reduces Large Pores,Anti-Ag...","[0.02513036, -0.08636747, 0.0073359767, -0.032...","[-0.03271273, 0.097567245, 0.06490646, 0.08879...",0.424459
880,881,Sekkisei Clear Wellness UV Defense Milk SPF 50...,"Water,Cyclomethicone,Zinc Oxide,Alcohol Denat....","Acne Trigger,Irritating,Eczema","[-0.014256903, -0.0066632787, 0.003123713, -0....","[-0.059598684, 0.1010493, 0.037541907, 0.04550...",0.559910
1189,1190,After Party Smoothing Cream,"Water,Dimethicone,Cyclopentasiloxane,Dimethico...","Reduces Irritation,Acne Trigger","[-0.010939268, -0.07772424, 0.042471122, -0.01...","[-0.03521848, 0.0350736, 0.029098554, 0.098265...",0.513339


## Both

In [84]:
from sklearn.metrics.pairwise import cosine_similarity



query = "pls recommend a skincare for me have Nodule 10 spot and inflammatory 5 spot acne and I have skin type of Normal"

query_vector = embedder.encode(query).reshape(1, -1)

cosine_similarities_ingridients = cosine_similarity(query_vector, df["ingridients_vector"].tolist())
cosine_similarities_afterUse = cosine_similarity(query_vector, df["afterUse_vector"].tolist())

df["similarity_score_ingridients"] = cosine_similarities_ingridients.flatten()
df["similarity_score_afterUse"] = cosine_similarities_afterUse.flatten()

# Combine the similarity scores
df["similarity_score"] = df[["similarity_score_ingridients", "similarity_score_afterUse"]].mean(axis=1)

# Sort by combined similarity score and return the most similar rows
df_sorted = df.sort_values(by="similarity_score", ascending=False)

ids = df_sorted.head(10)['id'].tolist()

ids


[10524, 18642, 5946, 5965, 3527, 7657, 12077, 18006, 11688, 4579]

In [64]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np


ingridients_vectors = np.array(df['ingridients_vector'].tolist())
afterUse_vectors = np.array(df['afterUse_vector'].tolist())

X = np.concatenate([ingridients_vectors, afterUse_vectors], axis=1)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# กำหนดจำนวนกลุ่ม (k)
k = 37

# ใช้ K-Means เพื่อจัดกลุ่มข้อมูล
kmeans = KMeans(n_clusters=k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# แสดงผลลัพธ์การจัดกลุ่ม
print("Clustering Result:")
print(df[['id', 'cluster']])

# สร้าง Ground Truth โดยการใช้กลุ่มที่ได้จาก K-Means
ground_truth = {}
for cluster_id in df['cluster'].unique():
    ground_truth[cluster_id] = df[df['cluster'] == cluster_id]['id'].tolist()

print("\nGround Truth (Cluster Groups):")
# print(ground_truth)


df


Clustering Result:
          id  cluster
0          1       22
1          2       36
2          3       28
3          4       17
4          5       22
...      ...      ...
19045  19046        1
19046  19047       28
19047  19048       28
19048  19049       22
19049  19050       21

[17526 rows x 2 columns]

Ground Truth (Cluster Groups):


,id,name,ingridients,afterUse,ingridients_vector,afterUse_vector,similarity_score_ingridients,similarity_score_afterUse,similarity_score,cluster
0,1,Glycolic Acid 7% Toning Solution,"Water,Glycolic Acid,Rosa Damascena Flower Wate...","Good For Oily Skin,Skin Texture,Reduces Large ...","[-0.0142573025, -0.031158729, -0.012762374, -0...","[-0.042678863, 0.029420577, 0.06968989, 0.0630...",0.065194,0.455153,0.260173,22
1,2,Toleriane Hydrating Gentle Face Cleanser,"Water,Glycerin,Pentaerythrityl Tetraethylhexan...","Good For Oily Skin,Redness Reducing,Reduces Ir...","[0.031710535, -0.048401132, 0.034568764, -0.04...","[-0.058093365, -0.006683484, 0.027535623, 0.08...",0.059411,0.398390,0.228900,36
2,3,Niacinamide 10% + Zinc 1%,"Water,Niacinamide,Pentylene Glycol,Zinc PCA,Di...","Good For Oily Skin,Redness Reducing,Acne Fight...","[-0.029236695, -0.014860135, -0.008159622, -0....","[-0.0467925, -0.005378254, 0.048078034, 0.1013...",0.075028,0.511322,0.293175,28
3,4,Superfood Antioxidant Cleanser,"Water,Cocamidopropyl Hydroxysultaine,Sodium Co...","Redness Reducing,Reduces Irritation,Skin Textu...","[-0.011621266, -0.039093073, 0.012563803, -0.0...","[0.017273393, -0.0008645504, 0.03886996, 0.109...",0.081497,0.369128,0.225313,17
4,5,Low pH Good Morning Gel Cleanser,"Water,Cocamidopropyl Betaine,Sodium Lauroyl Me...","Good For Oily Skin,Reduces Irritation,Reduces ...","[0.0048684794, -0.02391357, 0.005042913, -0.02...","[-0.06704735, 0.02327355, 0.04347716, 0.063853...",0.116789,0.405019,0.260904,22
...,...,...,...,...,...,...,...,...,...,...
19045,19046,Hydrating Facial Cleanser,"Water,Glycerin,Cetearyl Alcohol,Peg-40 Stearat...","Redness Reducing,Anti-Aging,Scar Healing,Brigh...","[0.019111374, -0.061045658, 0.023297507, -0.03...","[-0.02097442, -0.0077653113, 0.05874233, 0.145...",0.081579,0.452830,0.267205,1
19046,19047,Ginseng Essence Water,"Panax Ginseng Root Water,Butylene Glycol,Glyce...","Good For Oily Skin,Redness Reducing,Reduces Ir...","[-0.005072252, -0.02875612, -0.0051193675, -0....","[-0.045459837, -0.014924622, 0.032847833, 0.09...",0.130695,0.410996,0.270846,28
19047,19048,PM Facial Moisturizing Lotion,"Water,Glycerin,Caprylic/Capric Triglyceride,Ni...","Good For Oily Skin,Redness Reducing,Anti-Aging...","[0.011581133, -0.07052795, 0.03465522, -0.0548...","[-0.06334534, 0.007528051, 0.03474403, 0.08571...",0.086827,0.489283,0.288055,28
19048,19049,AHA 30% + BHA 2% Peeling Solution,"Glycolic Acid,Water,Aloe Barbadensis Leaf Wate...","Good For Oily Skin,Reduces Irritation,Skin Tex...","[0.015771536, -0.017377969, -0.0027047764, -0....","[-0.036620915, 0.0359531, 0.050598644, 0.06205...",0.133463,0.421525,0.277494,22


In [91]:
# Initialize lists to store the results
precision_list = []
recall_list = []
f1_score_list = []

# Loop through each cluster
for cluster_id in range(k):
    # Filter the dataframe for the specified cluster
    cluster_df = df[df['cluster'] == cluster_id]
    ground_truth = cluster_df['id'].tolist()

    # Define the retrieved ids from the recommendations
    retrieved = set(ids)  # Results from the recommendations
    relevant = set(ground_truth)  # Actual relevant items

    # Initialize counters
    true_positives = 0

    # Calculate true positives
    for id in ids:
        if id in relevant:
            true_positives += 1

    # Calculate Precision@k
    precision_k = true_positives / len(ids)

    # Calculate Recall@k
    recall_k = true_positives / len(relevant)

    # Calculate F1 Score
    f1_score = 2 * (precision_k * recall_k) / (precision_k + recall_k) if (precision_k + recall_k) != 0 else 0

    # Append the results to the lists
    precision_list.append(precision_k)
    recall_list.append(recall_k)
    f1_score_list.append(f1_score)

    # Display the results for the current cluster
    print(f"Cluster {cluster_id}:")
    print(f"Precision@10: {precision_k:.2f}")
    print(f"Recall@10: {recall_k:.2f}")
    print(f"F1 Score: {f1_score:.2f}")
    print()

# Display the average results across all clusters
print(f"Average Precision@10: {sum(precision_list) / k:.2f}")
print(f"Average Recall@10: {sum(recall_list) / k:.2f}")
print(f"Average F1 Score: {sum(f1_score_list) / k:.2f}")

Cluster 0:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 1:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 2:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 3:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 4:
Precision@10: 0.50
Recall@10: 0.03
F1 Score: 0.06

Cluster 5:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 6:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 7:
Precision@10: 0.10
Recall@10: 0.00
F1 Score: 0.00

Cluster 8:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 9:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 10:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 11:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 12:
Precision@10: 0.10
Recall@10: 0.00
F1 Score: 0.01

Cluster 13:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 14:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cluster 15:
Precision@10: 0.00
Recall@10: 0.00
F1 Score: 0.00

Cl

In [90]:
df2 = pd.read_csv(
    'D:\\Work\\Project_S_models\\notebooks\\recommend\\archive\\datasheet.csv')

df2

,brand,name,type,country,ingridients,afterUse
0,The Ordinary,Glycolic Acid 7% Toning Solution,Toner,Canada,"Water,Glycolic Acid,Rosa Damascena Flower Wate...","Good For Oily Skin,Skin Texture,Reduces Large ..."
1,La Roche-Posay,Toleriane Hydrating Gentle Face Cleanser,Face Cleanser,France,"Water,Glycerin,Pentaerythrityl Tetraethylhexan...","Good For Oily Skin,Redness Reducing,Reduces Ir..."
2,The Ordinary,Niacinamide 10% + Zinc 1%,Facial Treatment,Canada,"Water,Niacinamide,Pentylene Glycol,Zinc PCA,Di...","Good For Oily Skin,Redness Reducing,Acne Fight..."
3,Youth To The People,Superfood Antioxidant Cleanser,Face Cleanser,United States,"Water,Cocamidopropyl Hydroxysultaine,Sodium Co...","Redness Reducing,Reduces Irritation,Skin Textu..."
4,COSRX,Low pH Good Morning Gel Cleanser,Face Cleanser,South Korea,"Water,Cocamidopropyl Betaine,Sodium Lauroyl Me...","Good For Oily Skin,Reduces Irritation,Reduces ..."
...,...,...,...,...,...,...
19045,CeraVe,Hydrating Facial Cleanser,Face Cleanser,Canada,"Water,Glycerin,Cetearyl Alcohol,Peg-40 Stearat...","Redness Reducing,Anti-Aging,Scar Healing,Brigh..."
19046,Beauty of Joseon,Ginseng Essence Water,Essence,South Korea,"Panax Ginseng Root Water,Butylene Glycol,Glyce...","Good For Oily Skin,Redness Reducing,Reduces Ir..."
19047,CeraVe,PM Facial Moisturizing Lotion,Night Moisturizer,Canada,"Water,Glycerin,Caprylic/Capric Triglyceride,Ni...","Good For Oily Skin,Redness Reducing,Anti-Aging..."
19048,The Ordinary,AHA 30% + BHA 2% Peeling Solution,Facial Treatment,Canada,"Glycolic Acid,Water,Aloe Barbadensis Leaf Wate...","Good For Oily Skin,Reduces Irritation,Skin Tex..."


In [3]:
import pandas as pd


df2 = pd.read_csv('D:\\Work\\Project_S_models\\notebooks\\recommend\\archive\\datasheet.csv')


for item in df2['type'].unique() : 
    print(item)

Toner
Face Cleanser
Facial Treatment
Serum
General Moisturizer
Sunscreen
Exfoliator
Face Makeup
Bath & Body
Makeup Remover
Day Moisturizer
Other Haircare
Shampoo
Fragrance
Hand Care
Conditioner
Lip Moisturizer
Eye Moisturizer
Sheet Mask
Eye Makeup
Tanning
Wet Mask
Emulsion
Overnight Mask
Makeup Applicator
Night Moisturizer
Lip Makeup
Oil
Essence
nan
Cheek Makeup
Nail Care
Lip Mask
Eye Mask
Other
Tool
False Eyelash
